# 04. Despliegue en la nube y filtro temporal

Objetivo de este notebook:

1. Re-entrenar rápidamente el modelo para tenerlo en memoria
2. Traducir el modelo local (scikit-learn) a un clasificador de Google Earth Engine (ee.Classifier).
3. Aplicar el modelo a la región completa del Gran Chaco para detectar errores causados por el clima.
4. Ejecutar el Filtro Inteligente: reemplazar las clasificaciones espurias de 2020 por el valor estable que tenían en 2019.
5. Visualizar el "Antes y Después".

In [1]:
import ee
import geemap
from geemap import ml
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

ee.Initialize()

# 1. Rápido re-entrenamiento (para que este notebook funcione de forma independiente)
df = pd.read_csv('data/dataset_etiquetado.csv')
nombres_bandas = ['classification_2019', 'temperature_2m', 'total_precipitation_sum']
X = df[nombres_bandas]
y = df['es_espurio']

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X, y)

print("1. Modelo re-entrenado localmente.")

# 2. MAGIA DE GEEMAP: Traducir el modelo a la nube de Google
# Convertimos los árboles de decisión de Python a texto, y luego a un clasificador de Earth Engine
arboles_string = ml.rf_to_strings(modelo_rf, nombres_bandas)
clasificador_ee = ml.strings_to_classifier(arboles_string)

print("2. Modelo traducido a Google Earth Engine con éxito.")

1. Modelo re-entrenado localmente.
2. Modelo traducido a Google Earth Engine con éxito.


Ahora tomaremos la imagen satelital del año de la sequía extrema (2020). Le pasaremos el clasificador inteligente para que pinte de rojo todos los píxeles que, según el clima, fueron clasificados erróneamente por el satélite.

In [2]:
# 1. Cargar los datos crudos originales
mapbiomas = ee.Image('projects/mapbiomas-argentina/assets/LAND-COVER/COLLECTION-2/GENERAL/CLASSIFICATION/FINAL_CLASSIFICATION/CHACO/CHACO-FINAL-v1')
clima = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

mapa_19 = mapbiomas.select('classification_2019')
mapa_20 = mapbiomas.select('classification_2020')

# Clima de 2020
clima_20 = clima.filterDate('2020-01-01', '2020-12-31').mean().select(['temperature_2m', 'total_precipitation_sum'])

# 2. Unir las capas y renombrarlas EXACTAMENTE como las variables con las que entrenamos al modelo
imagen_a_predecir = mapa_19.addBands(clima_20).rename(nombres_bandas)

# 3. Que el modelo prediga todo el mapa (Genera un mapa lleno de 1s y 0s)
mapa_espurio_ia = imagen_a_predecir.classify(clasificador_ee)

# 4. El filtro:
# Lógica: Dónde el modelo dice que hay un error (1), reemplaza el mapa de 2020 con el bosque real de 2019. 
# Dónde dice que es normal (0), mantiene el mapa de 2020 original.
mapa_20_corregido = mapa_20.where(mapa_espurio_ia.eq(1), mapa_19)

print("Filtro inteligente aplicado al mapa. Preparando visualización...")

Filtro inteligente aplicado al mapa. Preparando visualización...


## Visualización Interactiva 

El mapa interactivo mostrará tres capas. Se pueden encender y apagar desde el ícono de capas (arriba a la derecha en el mapa) para visualizar la zona donde el modelo corrigió los errores.

In [7]:
# ==========================================
# 1. PALETA OFICIAL MAPBIOMAS ARGENTINA
# ==========================================
paleta_mb = ['#ffffff'] * 40
paleta_mb[3]  = '#006400'  # Bosque Nativo 
paleta_mb[4]  = '#00ff00'  # Formación Savánica 
paleta_mb[11] = '#45c2a5'  # Humedales 
paleta_mb[12] = '#b8af4f'  # Pastizales nativos 
paleta_mb[15] = '#ffd966'  # Pasturas 
paleta_mb[19] = '#e974ed'  # Agricultura 
paleta_mb[21] = '#ffefc3'  # Mosaico Agricultura/Pastura 
paleta_mb[24] = '#d3212d'  # Infraestructura Urbana 
paleta_mb[25] = '#b35a2b'  # Suelo Desnudo / No Vegetado 
paleta_mb[30] = '#9c0027'  # Minería 
paleta_mb[33] = '#0000ff'  # Cuerpos de agua 

vis_params_mb = {'min': 0, 'max': 39, 'palette': paleta_mb}

# ==========================================
# 2. FIGURA MACRO: LA PORTADA CIAN (CON LEYENDA)
# ==========================================
# Zoom alejado para ver gran parte del Chaco
Map_Macro = geemap.Map(center=[-26.7, -61.5], zoom=8)
Map_Macro.addLayer(mapa_20, vis_params_mb, 'Fondo 2020')
Map_Macro.addLayer(mapa_espurio_ia.selfMask(), {'min': 1, 'max': 1, 'palette': ['#00FFFF']}, 'Zonas Corregidas (Cian)')

# Crear y añadir el diccionario para la leyenda visual
leyenda_portada = {
    'Bosque Nativo': '006400',
    'Suelo Desnudo / Sequía': 'b35a2b',
    'Zonas Corregidas por filtro': '00FFFF'
}
Map_Macro.add_legend(legend_title="Filtro de Ruido Óptico (Gran Chaco, 2020)", legend_dict=leyenda_portada)

print("Mapa general con zonas corregidas en cian")
display(Map_Macro)

# ==========================================
# 3. FIGURAS MICRO: EL LADO A LADO
# ==========================================
# Zoom muy cercano (nivel 14 o 15) para ver los píxeles (30x30m)
Map_Micro = geemap.Map(center=[-26.7, -61.5], zoom=14)

# Cargamos AMBAS capas en el mismo mapa
Map_Micro.addLayer(mapa_20, vis_params_mb, '1. Original (Con Errores)', True) # True = Encendida por defecto
Map_Micro.addLayer(mapa_20_corregido, vis_params_mb, '2. Corregido', False) # False = Apagada por defecto

print("Mapa con capas 2020 original y corregida por el modelo")
display(Map_Micro)

Mapa general con zonas corregidas en cian


Map(center=[-26.7, -61.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

Mapa con capas 2020 original y corregida por el modelo


Map(center=[-26.7, -61.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [4]:
# 1. Definir explícitamente el Área de Interés (AOI) que usamos en el Notebook 2
aoi = ee.Geometry.Rectangle([-63.0, -27.5, -61.0, -26.0])

# 2. Aumentar la escala de reducción para evitar el "Time out"
# 300 metros es un tamaño bueno para estimar áreas masivas rápidamente
escala_calculo = 300 

print("Calculando áreas en la nube... esto tomará unos segundos.")

# Calcular el área total evaluada en el AOI
area_total = mapa_20.mask().multiply(ee.Image.pixelArea())
total_chaco_ha = area_total.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=escala_calculo,
    maxPixels=1e13
).getInfo()

# Calcular el área de los píxeles espurios (manchas rojas) que el modelo corrigió
area_espuria = mapa_espurio_ia.eq(1).multiply(ee.Image.pixelArea())
area_corregida_ha = area_espuria.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=escala_calculo,
    maxPixels=1e13
).getInfo()

# Convertir a hectáreas y calcular porcentaje
hectareas_arregladas = list(area_corregida_ha.values())[0] / 10000
hectareas_totales = list(total_chaco_ha.values())[0] / 10000
porcentaje = (hectareas_arregladas / hectareas_totales) * 100

print(f"Hectáreas recuperadas/corregidas en el AOI: {hectareas_arregladas:,.2f} ha")
print(f"Porcentaje de ruido óptico corregido: {porcentaje:.2f}%")

Calculando áreas en la nube... esto tomará unos segundos.
Hectáreas recuperadas/corregidas en el AOI: 579,616.19 ha
Porcentaje de ruido óptico corregido: 17.53%
